# Enrichment - Google Safe Browsing

Checks each URL against Google's list of known phishing/malware sites.
Free for non-commercial use. Get a key at:
https://console.cloud.google.com/ -> APIs & Services -> Library ->
search 'Safe Browsing API' -> Enable -> Credentials -> Create API Key

Add it to your .env file as:
SAFE_BROWSING_KEY=your_key_here

In [1]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

In [2]:

load_dotenv()
API_KEY = os.getenv('SAFE_BROWSING_KEY')

SAFE_BROWSING_URL = f'https://safebrowsing.googleapis.com/v4/threatMatches:find?key={API_KEY}'

print('Key loaded:', API_KEY is not None)

Key loaded: True


## Function to check a batch of URLs at once
The API supports checking many URLs in a single request -- much faster
than one request per site. Kept to batches of 5000 (a safe, conservative
size well under any documented limit).

In [5]:
def check_urls_batch(urls):
    """
    Checks a list of URLs against Google Safe Browsing.
    Returns a set of URLs that were flagged as unsafe.
    URLs NOT in the response are considered safe (the API only returns matches).
    """
    body = {
        'client': {
            'clientId': 'privacy-risk-scorer',
            'clientVersion': '1.0.0'
        },
        'threatInfo': {
            'threatTypes': ['MALWARE', 'SOCIAL_ENGINEERING', 'UNWANTED_SOFTWARE', 'POTENTIALLY_HARMFUL_APPLICATION'],
            'platformTypes': ['ANY_PLATFORM'],
            'threatEntryTypes': ['URL'],
            'threatEntries': [{'url': u} for u in urls]
        }
    }

    try:
        response = requests.post(SAFE_BROWSING_URL, json=body, timeout=15)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print('Batch failed:', e)
        return set()

    flagged = set()
    for match in data.get('matches', []):
        flagged.add(match['threat']['url'])

    return flagged

## Quick test on a few known-safe sites + one Google test URL
Google provides a permanent test URL that's ALWAYS flagged, specifically
so developers can confirm their integration works correctly.

In [6]:
test_urls = [
    'https://www.google.com',
    'https://www.wikipedia.org',
    'http://testsafebrowsing.appspot.com/s/malware.html'
]

flagged = check_urls_batch(test_urls)
print('Flagged URLs:', flagged)

Flagged URLs: {'http://testsafebrowsing.appspot.com/s/malware.html'}


## Run on your full URL list, in batches of 500

In [8]:
df_urls = pd.read_csv('../clean_data/urls_new.csv')
url_list = df_urls['url'].tolist()
print(f'Total URLs to check: {len(url_list)}')

Total URLs to check: 5000


In [9]:
all_flagged = set()
batch_size = 500

for i in range(0, len(url_list), batch_size):
    batch = url_list[i:i + batch_size]
    flagged = check_urls_batch(batch)
    all_flagged.update(flagged)
    print(f'Checked {min(i + batch_size, len(url_list))} / {len(url_list)} -- flagged so far: {len(all_flagged)}')
    time.sleep(1)

print()
print('Total flagged as unsafe:', len(all_flagged))
print(all_flagged)

Checked 500 / 5000 -- flagged so far: 0
Checked 1000 / 5000 -- flagged so far: 0
Checked 1500 / 5000 -- flagged so far: 0
Checked 2000 / 5000 -- flagged so far: 0
Checked 2500 / 5000 -- flagged so far: 0
Checked 3000 / 5000 -- flagged so far: 0
Checked 3500 / 5000 -- flagged so far: 0
Checked 4000 / 5000 -- flagged so far: 1
Checked 4500 / 5000 -- flagged so far: 1
Checked 5000 / 5000 -- flagged so far: 1

Total flagged as unsafe: 1
{'https://360safe.com'}


## Save as a simple enrichment table

In [11]:
df_urls['flagged_unsafe'] = df_urls['url'].isin(all_flagged)

print(df_urls['flagged_unsafe'].value_counts())

df_urls[['url', 'flagged_unsafe']].to_csv('../raw_data/safe_browsing_results.csv', index=False)
print('Saved safe_browsing_results.csv')

flagged_unsafe
False    4999
True        1
Name: count, dtype: int64
Saved safe_browsing_results.csv
